# Netflix Customer Churn & Revenue Insights Dashboard

This notebook performs exploratory data analysis on a cleaned Netflix-style streaming subscription churn dataset. The goal is to identify churn drivers, quantify revenue loss, and prepare insights for a Tableau or Power BI dashboard. The dataset is synthetic and is not official Netflix data.

## 1. Import Libraries and Load Data

In [ ]:
# pathlib helps us build file paths that work on Windows, macOS, and Linux.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Use a clean chart style for all plots in this notebook.
sns.set_theme(style="whitegrid", palette="deep")

# If the notebook is opened from the notebooks folder, the project root is one level up.
current_path = Path.cwd()
project_root = current_path.parent if current_path.name == "notebooks" else current_path

data_path = project_root / "data" / "processed" / "cleaned_customer_churn.csv"
df = pd.read_csv(data_path)

# Create a numeric churn flag so churn rate can be calculated with an average.
df["churn_flag"] = (df["churn_status"] == "Churned").astype(int)
df.head()

## 2. Dataset Overview

In [ ]:
# Show the number of rows, columns, and data types.
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.info()

In [ ]:
# Summary statistics help us understand ranges, averages, and possible outliers.
df.describe(include="all")

## 3. Missing Value Analysis

In [ ]:
# The cleaned dataset should have no missing values after the pipeline runs.
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## 4. Overall Churn Rate

In [ ]:
# Churn rate is the percentage of customers whose churn_status equals Churned.
total_customers = len(df)
churned_customers = int(df["churn_flag"].sum())
churn_rate = df["churn_flag"].mean() * 100

print(f"Total customers: {total_customers:,}")
print(f"Churned customers: {churned_customers:,}")
print(f"Overall churn rate: {churn_rate:.2f}%")

## 5. Churn by Contract Type

In [ ]:
# Group by contract type to compare churn risk across contract lengths.
contract_churn = (
    df.groupby("contract_type", as_index=False)
    .agg(customers=("customer_id", "count"), churn_rate=("churn_flag", "mean"))
)
contract_churn["churn_rate"] = contract_churn["churn_rate"] * 100
contract_churn = contract_churn.sort_values("churn_rate", ascending=False)
contract_churn

In [ ]:
# Bar charts make it easy to compare churn rates by category.
plt.figure(figsize=(8, 5))
sns.barplot(data=contract_churn, x="contract_type", y="churn_rate", color="#2F80ED")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

## 6. Churn by Payment Method

In [ ]:
# Payment method can reveal operational or customer behavior patterns.
payment_churn = (
    df.groupby("payment_method", as_index=False)
    .agg(customers=("customer_id", "count"), churn_rate=("churn_flag", "mean"))
)
payment_churn["churn_rate"] = payment_churn["churn_rate"] * 100
payment_churn = payment_churn.sort_values("churn_rate", ascending=False)
payment_churn

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=payment_churn, x="payment_method", y="churn_rate", color="#27AE60")
plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 7. Churn by Tenure Group

In [ ]:
# Tenure groups show whether churn is concentrated among new or long-time customers.
tenure_order = ["0-6 months", "7-12 months", "13-24 months", "25-48 months", "49+ months"]
tenure_churn = (
    df.groupby("tenure_group", as_index=False)
    .agg(customers=("customer_id", "count"), churn_rate=("churn_flag", "mean"))
)
tenure_churn["tenure_group"] = pd.Categorical(tenure_churn["tenure_group"], tenure_order, ordered=True)
tenure_churn["churn_rate"] = tenure_churn["churn_rate"] * 100
tenure_churn = tenure_churn.sort_values("tenure_group")
tenure_churn

In [ ]:
plt.figure(figsize=(9, 5))
sns.lineplot(data=tenure_churn, x="tenure_group", y="churn_rate", marker="o", linewidth=2)
plt.title("Churn Rate by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

## 8. Monthly Revenue Loss

In [ ]:
# Revenue loss estimates the MRR lost from customers who churned.
total_mrr = df["monthly_charges"].sum()
monthly_revenue_loss = df["monthly_revenue_loss"].sum()
annual_revenue_loss = df["annual_revenue_loss"].sum()

print(f"Total monthly recurring revenue: ${total_mrr:,.2f}")
print(f"Lost monthly recurring revenue: ${monthly_revenue_loss:,.2f}")
print(f"Estimated annual revenue loss: ${annual_revenue_loss:,.2f}")

In [ ]:
# Compare revenue loss across tenure groups.
revenue_by_tenure = (
    df.groupby("tenure_group", as_index=False)["monthly_revenue_loss"]
    .sum()
)
revenue_by_tenure["tenure_group"] = pd.Categorical(revenue_by_tenure["tenure_group"], tenure_order, ordered=True)
revenue_by_tenure = revenue_by_tenure.sort_values("tenure_group")

plt.figure(figsize=(9, 5))
sns.barplot(data=revenue_by_tenure, x="tenure_group", y="monthly_revenue_loss", color="#F2994A")
plt.title("Monthly Revenue Loss by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Monthly Revenue Loss ($)")
plt.tight_layout()
plt.show()

## 9. Key Takeaways

- Month-to-month customers have the highest churn rate and contribute the largest share of revenue loss.
- Customers in their first year show much higher churn risk than long-tenure customers.
- Electronic check customers have the highest payment-method churn rate in this sample.
- Competitor offers and price concerns are the most common recorded churn reasons.